# 07 — M1 Gate Decision

This notebook is the only authority for closing M1. It consumes persisted evidence from notebooks `00`–`06`; it does not infer success from notebook execution alone.

**Run order before this notebook:** `00 → 01 → 02 → 03 → 04 → 05 → 06 → 07`.


In [1]:
from pathlib import Path
import tomllib
import pandas as pd

from argos.common.paths import ProjectPaths

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()

with (paths.configs / "project.toml").open("rb") as handle:
    PROJECT_CONFIG = tomllib.load(handle)

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 160)


In [2]:
from argos.common.reports import write_report
from argos.evaluation.m1_gate import evaluate_m1_gate, render_m1_gate_markdown

gate_report, decision = evaluate_m1_gate(paths.reports, PROJECT_CONFIG)

write_report(
    gate_report,
    paths.reports / "profiling" / "m1_gate_report.parquet",
)

decision_path = paths.docs / "decisions" / "M1_GATE.md"
decision_path.parent.mkdir(parents=True, exist_ok=True)
decision_path.write_text(
    render_m1_gate_markdown(gate_report, decision),
    encoding="utf-8",
)

gate_report[[
    "check_id", "category", "passed", "observed", "requirement"
]]


,check_id,category,passed,observed,requirement
0,inventory_complete,inventory,True,"datasets=7, nonempty=7",>= 7 non-empty datasets
1,bronze_structural_integrity,inventory,True,ragged_rows=0,ragged_row_count = 0 across active snapshots
2,contracts_primary_key,grain,True,"{'null_rate': np.float64(0.0), 'uniqueness': np.float64(1.0)}",id_contrato null_rate=0 and uniqueness=1
3,suppliers_primary_key,grain,True,"{'null_rate': np.float64(0.0), 'uniqueness': np.float64(1.0)}",codigo null_rate=0 and uniqueness=1
4,processes_grain_characterized,grain,True,"{'selected_key': 'id_del_portafolio', 'interpretation': 'multiple_payload_variants_per_key', 'duplicate_keys': np.int64(222287), 'conflicting_keys': np.int6...",selected relationship key has a duplicate/payload diagnostic
5,additions_grain_characterized,grain,True,"{'interpretation': 'multiple_payload_variants_per_key', 'duplicate_excess': np.int64(19741938)}",event-key duplication diagnostic exists
6,contracts_processes_resolved,relationships,True,"{'right_keys': array(['id_del_portafolio'], dtype=object), 'coverage': 0.9754900910375112, 'requires_canonicalization': np.True_}",coverage >= 90.00%
7,relationship_contracts_suppliers,relationships,True,"{'coverage': 0.9466465459975967, 'expansion': np.float64(1.0), 'right_multiplicity': np.float64(1.0)}",coverage >= 90.00%
8,relationship_additions_contracts,relationships,True,"{'coverage': 0.9999459804294814, 'expansion': np.float64(1.0), 'right_multiplicity': np.float64(1.0)}",coverage >= 99.00%
9,relationship_bidders_processes,relationships,True,"{'coverage': 0.9883179072623022, 'expansion': np.float64(4.5650232092414), 'right_multiplicity': np.float64(8840.0)}",coverage >= 95.00%


In [3]:
status = "M1 ✅ CLOSED" if decision.closed else "M1 ❌ OPEN"
print(status)
print(f"Blocking checks: {decision.passed}/{decision.total}")
if decision.failed_checks:
    print("Failed:", ", ".join(decision.failed_checks))
print("Decision artifact:", decision_path)

assert decision.closed, (
    "M1 remains OPEN. Resolve the failed checks above and rerun the affected "
    "notebooks before starting Silver."
)


M1 ✅ CLOSED
Blocking checks: 19/19
Decision artifact: /Users/macm1pro/Desktop/projects/argos/argos-code/docs/decisions/M1_GATE.md


## What CLOSED means

Closure is scoped to the first **addition-risk V1**. It means the core contract/addition representation, target timestamps, key relationships, duplicate behavior and leakage register are sufficiently characterized to design Silver. It does **not** make `payments` or `sanctions` valid modeling inputs, and it does not waive point-in-time correctness.
